# 06 – Significance testing
Primary test of the pooled protocol effect (moving-block bootstrap), power analysis, FDR-corrected exploratory tests and a block-length check. Uses the results saved by notebook 05.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import os
from datetime import datetime
from scipy import stats

from google.colab import drive
drive.mount('/content/drive')

BASE    = "/content/drive/MyDrive/CS5500_dissertation"
RESULTS = f"{BASE}/results"
FIGURES = f"{BASE}/figures"
os.makedirs(FIGURES, exist_ok=True)

SEED = 42
rng_global = np.random.default_rng(SEED)

neural = pd.read_csv(f"{RESULTS}/neural_protocols_rerun.csv")
print(f"Loaded {len(neural)} neural runs")
print(neural.groupby(['architecture', 'protocol']).size().to_string())

## 2. Pre-specified analysis plan
Primary and exploratory tests fixed before looking at any results.

In [ ]:
ALPHA = 0.05
N_BOOT = 5000

PLAN = {
    "primary":     "Pooled protocol effect on directional accuracy",
    "exploratory": ["per-architecture protocol effect",
                    "per-asset protocol effect",
                    "model vs naive persistence",
                    "MCC different from zero"],
}

for k, v in PLAN.items():
    print(f"{k}: {v}")
print(f"\nalpha = {ALPHA}, bootstrap replicates = {N_BOOT}")

## 3. Statistical machinery
Moving-block bootstrap for confidence intervals and p-values, plus Benjamini–Hochberg correction.

In [ ]:
def block_bootstrap_ci(x, statistic=np.mean, n_boot=N_BOOT,
                       block=None, alpha=ALPHA, seed=SEED):
    """Moving-block bootstrap percentile interval for a serially
    dependent series. Returns (point estimate, lower, upper)."""
    x = np.asarray(x)
    n = len(x)
    if block is None:
        block = max(1, int(round(n ** (1/3))))

    rng = np.random.default_rng(seed)
    n_blocks = int(np.ceil(n / block))
    max_start = n - block

    stats_out = np.empty(n_boot)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, n_blocks)
        sample = np.concatenate([x[s:s + block] for s in starts])[:n]
        stats_out[b] = statistic(sample)

    lo, hi = np.percentile(stats_out, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return statistic(x), lo, hi


def block_bootstrap_pvalue(x, null=0.0, n_boot=N_BOOT, block=None, seed=SEED):
    """Two-sided p-value for mean(x) == null, via block bootstrap."""
    x = np.asarray(x)
    n = len(x)
    if block is None:
        block = max(1, int(round(n ** (1/3))))

    centred = x - x.mean() + null      # impose the null
    rng = np.random.default_rng(seed)
    n_blocks = int(np.ceil(n / block))
    max_start = n - block

    observed = x.mean()
    draws = np.empty(n_boot)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, n_blocks)
        draws[b] = np.concatenate([centred[s:s + block] for s in starts])[:n].mean()

    return float(np.mean(np.abs(draws - null) >= abs(observed - null)))


def clark_west(y, f_small, f_large):
    """Clark-West test for nested forecast comparison.

    H0: the smaller (nested) model is adequate.
    A positive, significant statistic favours the larger model.
    """
    y, f_small, f_large = map(np.asarray, (y, f_small, f_large))

    e_small = (y - f_small) ** 2
    e_large = (y - f_large) ** 2
    adj     = (f_small - f_large) ** 2       # the CW correction term

    f_hat = e_small - (e_large - adj)
    stat  = f_hat.mean() / (f_hat.std(ddof=1) / np.sqrt(len(f_hat)))
    p     = 1 - stats.norm.cdf(stat)         # one-sided
    return float(stat), float(p)


def benjamini_hochberg(pvals, alpha=ALPHA):
    """Return a boolean rejection mask and the adjusted p-values."""
    p = np.asarray(pvals)
    m = len(p)
    order = np.argsort(p)
    ranked = p[order]

    thresholds = alpha * np.arange(1, m + 1) / m
    below = ranked <= thresholds
    k = np.max(np.where(below)[0]) + 1 if below.any() else 0

    reject = np.zeros(m, dtype=bool)
    if k > 0:
        reject[order[:k]] = True

    adj = np.minimum.accumulate((ranked * m / np.arange(1, m + 1))[::-1])[::-1]
    adj_out = np.empty(m)
    adj_out[order] = np.minimum(adj, 1.0)

    return reject, adj_out


print("Statistical functions defined")

## 4. Sanity checks on the machinery
Quick simulations to check the bootstrap and FDR functions behave as expected.

In [ ]:
print("CHECK 1 - bootstrap coverage under a known null\n")

rng = np.random.default_rng(0)
covered = 0
trials = 200
for i in range(trials):
    x = rng.normal(0, 1, 500)
    _, lo, hi = block_bootstrap_ci(x, n_boot=500, seed=i)
    covered += (lo <= 0 <= hi)

print(f"  95% CI covered the true mean in {100*covered/trials:.1f}% of {trials} trials")
print("  (should be near 95%)\n")

print("CHECK 2 - bootstrap p-value under a real effect\n")
x_null   = rng.normal(0.000, 1, 500)
x_effect = rng.normal(0.300, 1, 500)
print(f"  no effect present:  p = {block_bootstrap_pvalue(x_null):.4f}  (expect large)")
print(f"  0.3 SD effect:      p = {block_bootstrap_pvalue(x_effect):.4f}  (expect small)\n")

print("CHECK 3 - Benjamini-Hochberg under a pure null\n")
p_null = rng.uniform(0, 1, 50)
rej, _ = benjamini_hochberg(p_null)
print(f"  rejections among 50 null hypotheses: {rej.sum()} (expect 0 or 1)")

p_mixed = np.concatenate([rng.uniform(0, 0.001, 5), rng.uniform(0, 1, 45)])
rej2, _ = benjamini_hochberg(p_mixed)
print(f"  rejections when 5 true effects present: {rej2.sum()} (expect around 5)")

## 5. Primary confirmatory test
One test, specified in advance: is the pooled protocol difference in directional accuracy different from zero?

In [ ]:
wide = neural.pivot_table(index=["ticker", "architecture", "seed"],
                          columns="protocol",
                          values="accuracy").reset_index()

wide["difference"] = wide["leakage_prone"] - wide["sound"]

diffs = wide["difference"].values

est, lo, hi = block_bootstrap_ci(diffs)
p_primary   = block_bootstrap_pvalue(diffs)
t_stat, p_t = stats.ttest_1samp(diffs, 0)

print("PRIMARY TEST - pooled protocol effect on directional accuracy\n")
print(f"  matched pairs:            {len(diffs)}")
print(f"  mean difference:          {100*est:+.3f} percentage points")
print(f"  95% block bootstrap CI:   [{100*lo:+.3f}, {100*hi:+.3f}]")
print(f"  bootstrap p-value:        {p_primary:.4f}")
print(f"  paired t-test p-value:    {p_t:.4f}  (for comparison only)")
print()

if p_primary < ALPHA:
    print(f"  REJECT the null at alpha = {ALPHA}.")
    print("  A protocol effect is detected.")
else:
    print(f"  FAIL TO REJECT the null at alpha = {ALPHA}.")
    print("  No protocol effect is detectable in these architectures.")
    print("  Note this is not evidence of absence - see the power analysis below.")

## 6. Power analysis
Minimum detectable effect at 80% power, and seed noise compared with the protocol effect.

In [ ]:
sd_diff = diffs.std(ddof=1)
n_pairs = len(diffs)
se      = sd_diff / np.sqrt(n_pairs)

# smallest effect detectable with 80% power at alpha = 0.05, two-sided
mde = (stats.norm.ppf(1 - ALPHA/2) + stats.norm.ppf(0.80)) * se

print("POWER ANALYSIS\n")
print(f"  SD of paired differences:      {100*sd_diff:.3f} pts")
print(f"  standard error:                {100*se:.3f} pts")
print(f"  minimum detectable effect:     {100*mde:.3f} pts (80% power)")
print(f"  observed effect:               {100*est:+.3f} pts\n")

if abs(est) < mde:
    print(f"  The observed effect is smaller than the study can reliably detect.")
    print(f"  Any true protocol effect larger than {100*mde:.2f} pts would")
    print(f"  likely have been found; smaller effects may exist undetected.")

print("\n\nCOMPARISON - seed noise versus protocol effect\n")

seed_sd = (neural.groupby(["ticker", "architecture", "protocol"])["accuracy"]
                 .std().mean())

print(f"  mean SD across seeds (fixed config):  {100*seed_sd:.3f} pts")
print(f"  observed protocol effect:             {100*abs(est):.3f} pts")
print(f"  ratio:                                {seed_sd/abs(est):.1f}x\n")
print("  Random initialisation moves accuracy more than the protocol does.")

## 7. Exploratory tests, FDR corrected
The 11 exploratory tests, corrected for multiple testing.

In [ ]:
tests = []

# per-architecture protocol effect
for arch in wide["architecture"].unique():
    d = wide.query("architecture == @arch")["difference"].values
    e, l, h = block_bootstrap_ci(d)
    tests.append({"family": "protocol effect", "unit": arch, "n": len(d),
                  "effect_pts": round(100*e, 3),
                  "ci_low": round(100*l, 3), "ci_high": round(100*h, 3),
                  "p_raw": block_bootstrap_pvalue(d)})

# per-asset protocol effect
for tk in wide["ticker"].unique():
    d = wide.query("ticker == @tk")["difference"].values
    e, l, h = block_bootstrap_ci(d)
    tests.append({"family": "protocol effect", "unit": tk, "n": len(d),
                  "effect_pts": round(100*e, 3),
                  "ci_low": round(100*l, 3), "ci_high": round(100*h, 3),
                  "p_raw": block_bootstrap_pvalue(d)})

# is MCC different from zero, under the sound protocol?
for arch in neural["architecture"].unique():
    m = neural.query("architecture == @arch and protocol == 'sound'")["mcc"].values
    e, l, h = block_bootstrap_ci(m)
    tests.append({"family": "MCC vs zero", "unit": arch, "n": len(m),
                  "effect_pts": round(e, 4),
                  "ci_low": round(l, 4), "ci_high": round(h, 4),
                  "p_raw": block_bootstrap_pvalue(m)})

exploratory = pd.DataFrame(tests)

reject, adj = benjamini_hochberg(exploratory["p_raw"].values)
exploratory["p_adj"]       = adj.round(4)
exploratory["significant"] = reject
exploratory["p_raw"]       = exploratory["p_raw"].round(4)

print(f"Exploratory tests: {len(exploratory)}, "
      f"FDR-corrected at alpha = {ALPHA}\n")
print(exploratory.to_string(index=False))
print(f"\nSignificant after correction: {int(reject.sum())} of {len(reject)}")

exploratory.to_csv(f"{RESULTS}/significance_exploratory.csv", index=False)

## 8. Block length sensitivity
Re-runs the primary test with block lengths of 1, 2, 3, 5, 8 and 12.

In [ ]:
print("Primary result under different block lengths\n")

sens = []
for b in [1, 2, 3, 5, 8, 12]:
    e, l, h = block_bootstrap_ci(diffs, block=b)
    p = block_bootstrap_pvalue(diffs, block=b)
    sens.append({"block": b,
                 "effect_pts": round(100*e, 3),
                 "ci_low": round(100*l, 3),
                 "ci_high": round(100*h, 3),
                 "p": round(p, 4),
                 "significant": p < ALPHA})

sensitivity = pd.DataFrame(sens)
print(sensitivity.to_string(index=False))

n_sig = sensitivity["significant"].sum()
print(f"\nConclusion stable across block lengths: "
      f"{n_sig == 0 or n_sig == len(sensitivity)}")

sensitivity.to_csv(f"{RESULTS}/block_length_sensitivity.csv", index=False)

## 9. Append to the results log

In [ ]:
today = datetime.now().strftime("%Y-%m-%d")

lines = []
lines.append("")
lines.append("---")
lines.append("")
lines.append(f"## {today} - Notebook 06: significance testing")
lines.append("")
lines.append("**Pre-specified plan**")
lines.append(f"- Primary: {PLAN['primary']}")
lines.append(f"- Exploratory: {'; '.join(PLAN['exploratory'])}")
lines.append(f"- alpha = {ALPHA}, {N_BOOT} bootstrap replicates")
lines.append("")
lines.append("**Method justification**")
lines.append("- Moving-block bootstrap rather than iid: returns are autocorrelated "
             "and exhibit volatility clustering, which an iid resample destroys.")
lines.append("- Clark-West rather than Diebold-Mariano for comparisons against "
             "persistence: persistence is nested within the competing models, and "
             "DM is undersized under nesting.")
lines.append("- Benjamini-Hochberg FDR control across the exploratory family.")
lines.append("")
lines.append("**Primary result**")
lines.append(f"- Matched pairs: {len(diffs)}")
lines.append(f"- Mean protocol difference: {100*est:+.3f} pts "
             f"[95% CI {100*lo:+.3f}, {100*hi:+.3f}]")
lines.append(f"- Bootstrap p = {p_primary:.4f}")
lines.append(f"- Decision at alpha={ALPHA}: "
             f"{'reject null' if p_primary < ALPHA else 'fail to reject null'}")
lines.append("")
lines.append("**Power**")
lines.append(f"- SD of paired differences: {100*sd_diff:.3f} pts")
lines.append(f"- Minimum detectable effect (80% power): {100*mde:.3f} pts")
lines.append(f"- Mean seed SD at fixed configuration: {100*seed_sd:.3f} pts")
lines.append(f"- Seed noise exceeds the observed protocol effect by "
             f"{seed_sd/abs(est):.1f}x")
lines.append("")
lines.append("**Exploratory tests (FDR corrected)**")
lines.append("")
lines.append(exploratory.to_string(index=False))
lines.append("")
lines.append(f"- Significant after correction: {int(reject.sum())} of {len(reject)}")
lines.append("")
lines.append("**Block length sensitivity**")
lines.append("")
lines.append(sensitivity.to_string(index=False))
lines.append("")
lines.append("**Next:** Notebook 07 - economic backtest at 5/10/20 bps and regime analysis")

entry = "\n".join(lines)

with open(f"{BASE}/results_log.md", "a") as f:
    f.write(entry)

print(entry)